# Early Warning System beban CPU tinggi: peringatan 2 jam sebelum CPU ≥ 80 %

Notebook ini **berdiri sendiri** dan menjadi pasangan `skripsi_forecast_v3.ipynb` (1 jam), `skripsi_forecast_30m.ipynb`, dan `skripsi_forecast_15m.ipynb`. Ganti `RES` di sel konfigurasi (60, 30, atau 15), lalu *Run All*. Hasil ditulis ke `outputs_ews_{RES}/`. Bagian 9 membandingkan ketiga resolusi bila folder hasilnya sudah ada.

**Pertanyaan.** Dari setiap jam (*origin* t) saat CPU masih < 80 %, apakah CPU per jam akan mencapai **≥ 80 % dalam 2 jam ke depan**? Model memberi probabilitas; alarm dibunyikan bila probabilitas ≥ ambang keputusan yang disetel di validation. Ambang kejadian (80 %) tetap; yang disetel hanya ambang probabilitas alarm.

**Kenapa ini masalah data tidak seimbang.** Kelas "akan tinggi" hanya sekitar 5 % origin di validation dan 2 % di test. Akurasi dan ROC-AUC menyesatkan pada kelas selangka ini (Chicco & Jurman, 2023), sehingga seleksi memakai PR-AUC dan penanganan ketidakseimbangan dibandingkan secara eksplisit.

**Protokol** sama dengan notebook forecasting: walk-forward refit bulanan, seleksi hanya di validation (Mei–Agu 2026), seleksi dibekukan dengan hash, test (Sep–Okt 2026) sekali. Label selalu dari CPU per jam resmi (`cleaned_unimputed.csv`), sehingga ketiga resolusi dinilai pada kejadian yang sama. Periode test sudah pernah dilihat di notebook forecasting, jadi hasil test bersifat **eksploratori**.

**Revisi v3 (11 Nov 2026)** — semua hasil lama dihapus; notebook perlu dijalankan ulang untuk RES = 60, 30, dan 15.

| Masalah di versi sebelumnya | Perbaikan |
|---|---|
| XGBoost memakai `min_child_weight = 50`. Di XGBoost parameter ini adalah jumlah **hessian** minimum per daun, bukan jumlah sampel; pada log-loss hessian = p(1 − p) yang sangat kecil untuk kelas langka, sehingga hampir semua split terlarang (PR-AUC "tanpa penanganan" 0,38 dan "under-sampling" 0,08, jauh di bawah keluarga lain) | `min_child_weight = 1` (bawaan); kompleksitas tetap dibatasi kedalaman 4 |
| Under-sampling paling buruk di keempat keluarga dan membuang ±95 % data mayoritas | Dihapus; tersisa empat strategi |
| Pembanding "prakiraan model final" dan "kuantil 0,90" hanya tersedia di origin tiap 6 jam (437 origin validation dengan 3 positif, 221 origin test dengan 1 positif), sehingga tidak bermakna | Dihapus |
| Seleksi = PR-AUC tertinggi, padahal selisih antarkonfigurasi teratas (0,609 vs 0,603) jauh di bawah ketidakpastian sampelnya | **Aturan satu galat baku**: di antara konfigurasi yang PR-AUC-nya dalam 1 SE (block bootstrap) dari yang terbaik, pilih yang paling sederhana (James dkk., 2021, §6.1.3) |
| Model tunggal saja | Ditambah **ensemble GBDT** (rata-rata probabilitas HGB, LightGBM, XGBoost), sama dengan model final forecasting (Mienye & Sun, 2022) |
| Alarm pada jam berturut-turut dihitung sebagai banyak alarm palsu | Dilaporkan juga **kejadian alarm**: alarm berturut-turut dihitung satu |
| Hasil test tanpa ketidakpastian (hanya 29 positif) | CI 95 % block bootstrap untuk PR-AUC dan selisihnya terhadap ambang statis |
| Bootstrap perbandingan resolusi memakai `np.isin`, sehingga hari yang terambil dua kali hanya dihitung sekali | Resample dengan pengembalian yang benar (fungsi `blok_bootstrap`) |
| SMOTE dan imputasi dihitung ulang untuk setiap keluarga model | Dihitung sekali per refit dan strategi (≈ 3× lebih cepat untuk strategi SMOTE) |
| Fitur | Ditambah kemiringan CPU 3 jam, jam sejak terakhir CPU ≥ 80 %, dan maks. CPU pada jendela yang sama kemarin |

### Langkah dan dasar pustaka (jurnal 2020–2026; daftar lengkap di akhir dan di `DASAR_PUSTAKA_EWS.md`)

| Langkah | Keputusan | Dasar |
|---|---|---|
| Formulasi EWS | Klasifikasi "kejadian dalam jendela L jam ke depan", dinilai time-based (latih sebelum, uji sesudah) | Li dkk. (2020); Hyland dkk. (2020) |
| Origin | Hanya jam dengan CPU < 80 % (peringatan harus *dini*), riwayat 168 jam lengkap, ≥ 95 % label jendela teramati | Hyland dkk. (2020); Hewamalage dkk. (2023) |
| Fitur | Lag, rolling, pola jam-minggu, anomali, kalender libur, telemetri terpilih, plus fitur sub-jam untuk data 30/15 menit | Januschowski dkk. (2022); Hyndman & Athanasopoulos (2021) |
| Model | Regresi logistik, HistGradientBoosting, LightGBM, XGBoost, ensemble GBDT (rata-rata probabilitas tiga boosting) | Hyland dkk. (2020); Tengku Asmawi dkk. (2022); Eldeeb dkk. (2026); Mienye & Sun (2022) |
| Ketidakseimbangan | Empat strategi: tanpa, bobot kelas (cost-sensitive), SMOTE, hibrida SMOTE + bobot; resampling hanya di training. "Tanpa penanganan" + ambang yang disetel tetap pembanding utama, karena koreksi ketidakseimbangan merusak kalibrasi probabilitas | Rezvani & Wang (2023); Thabtah dkk. (2020); Tarawneh dkk. (2022); Eldeeb dkk. (2026); Chawla dkk. (2002); van den Goorbergh dkk. (2022) |
| Ambang alarm | Disetel di validation (maks. F1), bukan 0,5 | Esposito dkk. (2021) |
| Metrik dan seleksi | Seleksi: PR-AUC dengan aturan satu galat baku (SE dari block bootstrap 7 hari). Dilaporkan: presisi, recall, F1, MCC, spesifisitas, ROC-AUC, Brier (kalibrasi) | Chicco & Jurman (2020, 2023); Hyland dkk. (2020); James dkk. (2021, §6.1.3) |
| Evaluasi kejadian | Persentase episode terdeteksi, lead time alarm pertama, kejadian alarm palsu per minggu; pembanding ambang statis; CI 95 % block bootstrap | Hyland dkk. (2020); Li dkk. (2020) |
| Pemisahan data | Urutan waktu dijaga, tanpa shuffle, label training harus matang sebelum refit | Cerqueira dkk. (2020); Hewamalage dkk. (2023) |

## 0. Konfigurasi

In [ ]:
RES             = 60            # resolusi data: 60 (1 jam), 30, atau 15 menit
FITUR           = "tele"        # "tele" = CPU + kalender + telemetri terpilih; "cpu" = CPU + kalender
LEADS           = [2]           # jendela peringatan (jam); boleh beberapa, mis. [2, 6]
RASIO           = 1.0           # rasio minoritas:mayoritas sesudah SMOTE (1.0 = seimbang)
RIWAYAT_PANJANG = False         # True: pakai seluruh riwayat data 30/15 menit (default: mulai sama dengan data 1 jam)
REUSE_CACHE     = True
OUT_DIR         = f"outputs_ews_{RES}"

LABEL_PATH  = "cleaned_unimputed.csv"                                   # label resmi per jam (sama untuk semua resolusi)
DATA_PATH   = {60: "cleaned_unimputed.csv", 30: "cleaned_30m_unimputed.csv", 15: "cleaned_15m_unimputed.csv"}
SEL60_PATH  = "outputs_v3_baru/seleksi_beku.json"                       # daftar telemetri terpilih (sama dengan notebook forecasting)
DATA_CUTOFF = "2026-11-01 00:00"
HIGH        = 80.0
SPLIT = {"validation": {"refits": ["2026-05-01", "2026-06-01", "2026-07-01", "2026-08-01"], "end": "2026-09-01"},
         "test":       {"refits": ["2026-09-01", "2026-10-01"], "end": DATA_CUTOFF}}
MAX_GAP_H, MIN_TARGET = 3, 0.95          # C1: celah <= 3 jam diinterpolasi (fitur saja); C2: label butuh >= 95 % jam teramati
N_ITER, LR, SEED, N_JOBS = 300, 0.05, 42, 6
K_SMOTE = 5
N_BOOT, BLOK_HARI = 500, 7               # block bootstrap: ulangan dan panjang blok (hari)
PIPELINE_VERSION = "3"                   # ikut kunci cache; naikkan bila logika pelatihan berubah

import os, json, time, pickle, hashlib, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.dates as mdates
import holidays, sklearn, lightgbm as lgb, xgboost as xgb
from IPython.display import display, Markdown
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve, brier_score_loss, matthews_corrcoef, confusion_matrix
warnings.filterwarnings("ignore")
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True); os.makedirs(f"{OUT_DIR}/cache", exist_ok=True)

TARGET = "cpu_usage_percent"
METRICS = ["cpu_usage_percent", "sysload_1m", "sysload_5m", "sysload_15m", "memory_usage_percent", "swap_usage_percent",
           "network_rx_bytes_per_sec", "network_tx_bytes_per_sec", "disk_read_bytes_per_sec", "disk_write_bytes_per_sec",
           "disk_read_iops", "disk_write_iops", "process_running_count", "process_blocked_count"]
TELE_DEFAULT = ["sysload_1m", "sysload_15m", "network_rx_bytes_per_sec", "network_tx_bytes_per_sec",
                "disk_read_bytes_per_sec", "disk_write_bytes_per_sec", "disk_read_iops", "disk_write_iops"]
LOG_COLS  = {c for c in METRICS if "bytes" in c or "iops" in c or "process" in c}
TELE_LONG = ("sysload_15m", "disk_read_iops")
FAM   = {"logit": "Regresi logistik", "hgb": "HistGradientBoosting", "lgbm": "LightGBM", "xgb": "XGBoost", "ens": "Ensemble GBDT"}
GBDT  = ("hgb", "lgbm", "xgb")                                   # anggota ensemble; "ens" tidak dilatih sendiri
STRAT = {"none": "Tanpa penanganan", "bobot": "Bobot kelas (cost-sensitive)", "smote": "SMOTE", "hibrida": "SMOTE + bobot kelas (hibrida)"}   # urutan = makin kompleks
STRAT_PENDEK = {"none": "tanpa", "bobot": "bobot", "smote": "SMOTE", "hibrida": "hibrida"}
FAM_URUT = {"logit": 0, "hgb": 1, "lgbm": 1, "xgb": 1, "ens": 2}  # kesederhanaan keluarga untuk aturan satu galat baku
BASEL = {"ambang_cpu": "Ambang statis CPU saat ini", "naif": "Naif musiman (maks. minggu lalu)"}
FS_NAME = {"cpu": "CPU + kalender", "tele": "CPU + kalender + telemetri terpilih"}
RES_NAME = {60: "1 jam", 30: "30 menit", 15: "15 menit"}
COL = {"logit": "#6250d6", "hgb": "#2a78d6", "lgbm": "#eb6834", "xgb": "#1baf7a", "ens": "#c98a00", "base": "#52514e",
       "r60": "#2a78d6", "r30": "#eb6834", "r15": "#1baf7a"}            # palet dicek dengan validator CVD
INK, INK2, GRIDC, BAND = "#0b0b0b", "#52514e", "#e6e5e0", "#f1f0eb"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRIDC,
                     "grid.linewidth": .6, "font.size": 9, "axes.titlesize": 10, "axes.titlelocation": "left", "legend.frameon": False})
FASE = {"validation": "Validation", "test": "Test"}
DEVICE  = os.environ.get("ML_DEVICE", "cpu")                 # "gpu": XGBoost CUDA
XGB_DEV = {"device": "cuda"} if DEVICE == "gpu" else {}
VERSIONS = {"holidays": holidays.__version__, "scikit-learn": sklearn.__version__, "lightgbm": lgb.__version__, "xgboost": xgb.__version__,
            "pandas": pd.__version__, "numpy": np.__version__, "device": f"xgb:{'cuda' if XGB_DEV else 'cpu'}"}
xgb.set_config(verbosity=0)
SPH = 60 // RES

def sha_file(p): return hashlib.sha256(open(p, "rb").read()).hexdigest()
def sha_obj(o): return hashlib.sha256(json.dumps(o, sort_keys=True, default=str).encode()).hexdigest()
def show(df, fmt=None, caption=None):
    st = df.style.hide(axis="index").format(fmt or None, na_rep="–", precision=3, escape="html")
    if caption: st = st.set_caption(caption).set_table_styles([{"selector": "caption", "props": "caption-side: top; font-weight: 600; text-align: left;"}])
    display(st)
KOLOM_METRIK = ["PR-AUC", "ROC-AUC", "Presisi", "Recall", "F1", "MCC", "Spesifisitas", "Brier"]

def md_table(df, floatfmt=".3f"):
    """Tabel markdown tanpa dependensi tabulate."""
    cols = list(df.columns); fmt = lambda v: (f"{v:{floatfmt}}" if isinstance(v, (float, np.floating)) and np.isfinite(v) else ("-" if isinstance(v, float) else str(v)))
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows(): lines.append("| " + " | ".join(fmt(r[c]) for c in cols) + " |")
    return "\n".join(lines)
show(pd.DataFrame({"Pustaka": list(VERSIONS), "Versi": list(VERSIONS.values())}), caption="Versi pustaka yang dipakai")
print(f"EWS data {RES_NAME[RES]} | fitur {FS_NAME[FITUR]} | jendela {LEADS} jam | rasio sampling {RASIO}")

## 1. Data

Label resmi per jam dibaca dari `cleaned_unimputed.csv` (timestamp di akhir interval: baris *t* = rata-rata (t − 1 jam, t]). Data resolusi `RES` dibaca dari file yang sesuai; untuk 30/15 menit riwayatnya dimulai sama dengan data 1 jam supaya ketiga varian sebanding (varian utama notebook 30m/15m). Baris sesudah **1 Nov 2026** tidak dibaca.

**Pembersihan (C1).** Celah ≤ 3 jam diinterpolasi linear **hanya untuk fitur**; celah lebih panjang dibiarkan kosong. Label tidak pernah diimputasi.

In [ ]:
def muat_label():
    y = pd.read_csv(LABEL_PATH, parse_dates=["timestamp"])
    assert y.timestamp.dt.tz is None
    y = y.drop_duplicates().set_index("timestamp").sort_index()[TARGET].astype(float)
    assert not y.index.duplicated().any(), "duplikat berkonflik di label"
    y = y[y.index < pd.Timestamp(DATA_CUTOFF)]
    y = y.reindex(pd.date_range(y.index[0], y.index[-1], freq="1h"))
    assert y.dropna().between(0, 100).all()
    return y
def muat_data(res, y_jam, riwayat_panjang):
    raw = pd.read_csv(DATA_PATH[res], parse_dates=["timestamp"])
    assert not (set(["timestamp", *METRICS]) - set(raw.columns)), "kolom hilang"
    assert raw.timestamp.dt.tz is None, "timestamp harus tanpa zona waktu"
    for c in METRICS: raw[c] = pd.to_numeric(raw[c], errors="raise").astype(float)
    raw = raw.drop_duplicates()
    assert not raw.timestamp.duplicated().any(), "duplikat berkonflik"
    raw = raw.set_index("timestamp").sort_index()
    assert raw.index.minute.isin(range(0, 60, res)).all() and (raw.index.second == 0).all(), "timestamp tidak di grid"
    raw = raw[raw.index < pd.Timestamp(DATA_CUTOFF)]
    if not riwayat_panjang:                                            # riwayat sama dengan data 1 jam (varian utama notebook 30m/15m)
        raw = raw[raw.index > y_jam.index[0] - pd.Timedelta(hours=1)]
    step = pd.Timedelta(minutes=res)
    df = raw.reindex(pd.date_range(raw.index[0], raw.index[-1], freq=step))
    assert df[TARGET].dropna().between(0, 100).all(), "CPU di luar 0-100"
    return df
def interpolasi_celah_pendek(d, max_steps):
    """C1: hanya celah <= max_steps yang diinterpolasi linear; celah lebih panjang dibiarkan kosong (outage)."""
    out = d.copy()
    for c in d.columns:
        s = d[c]; na = s.isna()
        if not na.any(): continue
        run = (na != na.shift()).cumsum(); runlen = na.groupby(run).transform("sum")
        ok = na & (runlen <= max_steps)
        out[c] = s.where(~ok, s.interpolate("linear", limit_area="inside"))
    return out

y_jam = muat_label(); df = muat_data(RES, y_jam, RIWAYAT_PANJANG)
DATA_SHA = sha_file(DATA_PATH[RES]); LABEL_SHA = sha_file(LABEL_PATH)
tele = list(json.load(open(SEL60_PATH))["telemetri_terpilih"]) if os.path.exists(SEL60_PATH) else TELE_DEFAULT
if FITUR == "cpu": tele = []
feat = interpolasi_celah_pendek(df[[TARGET] + tele], MAX_GAP_H * SPH)
show(pd.DataFrame([{"Data": RES_NAME[RES], "Awal": df.index[0], "Akhir": df.index[-1], "Baris": len(df), "CPU kosong (%)": 100 * df[TARGET].isna().mean(),
                    "CPU kosong sesudah C1 (%)": 100 * feat[TARGET].isna().mean(), "Jam label resmi": int(y_jam.notna().sum()), "Telemetri dipakai": len(tele)}]),
     {"Baris": "{:,d}", "Jam label resmi": "{:,d}", "Telemetri dipakai": "{:d}"}, "Tabel 1. Data dan pembersihan")

## 2. Kalender libur Indonesia

Dari library `holidays` (libur nasional dan cuti bersama), memakai tanggal **awal** interval. Fitur: libur, jarak ke/dari libur (maks. 14 hari), dan untuk jendela peringatan: apakah ada libur atau akhir pekan di dalam jendela.

In [ ]:
def kalender(index):
    years = list(range(index.year.min() - 1, index.year.max() + 2))
    pub = holidays.country_holidays("ID", years=years); gov = holidays.country_holidays("ID", years=years, categories=("government",))
    cal = pd.DataFrame(0.0, index=pd.date_range(f"{years[0]}-01-01", f"{years[-1]}-12-31"), columns=["libur_nasional", "cuti_bersama"])
    cal.loc[[d for d in cal.index if d in pub], "libur_nasional"] = 1.0
    cal.loc[[d for d in cal.index if d in gov], "cuti_bersama"] = 1.0
    off = (cal.libur_nasional + cal.cuti_bersama).gt(0) | (cal.index.dayofweek >= 5)
    hol = cal.index[(cal.libur_nasional + cal.cuti_bersama).gt(0)]
    nxt = pd.Series(hol, index=hol).reindex(cal.index).bfill(); prv = pd.Series(hol, index=hol).reindex(cal.index).ffill()
    cal["hari_ke_libur"] = ((nxt - cal.index).dt.days).clip(upper=14).fillna(14).astype(float)
    cal["hari_sejak_libur"] = ((cal.index - prv).dt.days).clip(upper=14).fillna(14).astype(float)
    cal["libur"] = (cal.libur_nasional + cal.cuti_bersama).gt(0).astype(float)
    return cal
cal = kalender(df.index)

## 3. Fitur, origin, dan uji kebocoran

Semua fitur hanya memakai data ≤ origin *t* (jam bulat).
- **Fitur dasar per jam** (dari rata-rata per jam): CPU saat ini, lag 1–168 jam, rolling 24/72/168 jam, kemiringan 1 dan 3 jam, jumlah jam ≥ 80 %, jam sejak terakhir CPU ≥ 80 % (kejadian cenderung berulang), tren, perubahan mingguan, profil jam-minggu ekspansif, anomali terhadap profil, telemetri terpilih (saat ini, rata-rata 24 jam, lag 24 jam), kalender origin.
- **Fitur sub-jam** (hanya data 30/15 menit): CPU langkah terakhir, lag sub-jam, selisih langkah, deviasi 2 jam, maks./min. 6 jam, selisih telemetri.
- **Fitur jendela** (t, t + L]: maks. CPU pada jendela yang sama kemarin (bila L ≤ 24 jam) dan 1–4 minggu lalu, maks. dan rata-rata profil jam-minggu di jendela, libur dan akhir pekan di jendela.

**Origin yang dinilai (C2 + syarat dini):** CPU resmi teramati di origin dan **< 80 %**, riwayat 169 jam lengkap sesudah C1, dan ≥ 95 % jam label di jendela teramati.

**Uji kebocoran:** fitur di origin *t* tidak boleh berubah bila seluruh data sesudah *t* diacak.

In [ ]:
def how(ix): s = ix - pd.Timedelta(hours=1); return (s.dayofweek * 24 + s.hour).to_numpy()
def profil_jam_minggu(c):
    """prof[i,k] = rata-rata CPU pada baris <= i dengan jam-minggu k (ekspansif, kausal)."""
    hw = how(c.index); v = c.to_numpy(float); ok = np.isfinite(v)
    oh = np.zeros((len(v), 168)); oh[np.arange(len(v))[ok], hw[ok]] = 1.0
    sums = np.cumsum(oh * np.where(ok, v, 0)[:, None], axis=0); cnt = np.cumsum(oh, axis=0)
    with np.errstate(invalid="ignore", divide="ignore"): return np.where(cnt > 0, sums / cnt, np.nan)
def fitur_dasar(feat, sph, tele):
    """Fitur di setiap jam (origin) dari data resolusi `sph` langkah per jam. Semua jendela hanya memakai data <= origin."""
    hourly_ix = feat.index[(feat.index.minute == 0)]
    def ke_jam(s): return s.rolling(sph, min_periods=sph).mean().reindex(hourly_ix)      # rata-rata (t - 1 jam, t]
    H = ke_jam(feat[TARGET]); out = {"cpu_now": H}
    for l in (1, 2, 3, 6, 12, 24, 48, 72, 168): out[f"cpu_lag_{l}"] = H.shift(l)
    for w in (24, 72, 168):
        r = H.rolling(w, min_periods=w); out[f"cpu_mean_{w}"], out[f"cpu_std_{w}"], out[f"cpu_max_{w}"], out[f"cpu_min_{w}"] = r.mean(), r.std(ddof=0), r.max(), r.min()
    out["cpu_diff_1"] = H - H.shift(1); out["cpu_diff_3"] = H - H.shift(3); out["cpu_diff_168"] = H - H.shift(168)
    hi = H.ge(HIGH).astype(float).where(H.notna())
    out["cpu_jam_ge80_24"] = hi.rolling(24, min_periods=24).sum(); out["cpu_jam_ge80_168"] = hi.rolling(168, min_periods=160).sum()
    urut = pd.Series(np.arange(len(H), dtype=float), index=H.index)                       # jam sejak terakhir CPU >= HIGH (maks. 168)
    out["jam_sejak_ge80"] = (urut - urut.where(H.ge(HIGH)).ffill()).clip(upper=168).fillna(168)
    m672 = H.rolling(672, min_periods=600).mean(); out["cpu_mean_672"] = m672; out["cpu_tren_168_672"] = out["cpu_mean_168"] - m672
    out["cpu_wow_24"] = out["cpu_mean_24"] - out["cpu_mean_24"].shift(168); out["cpu_wow_168"] = out["cpu_mean_168"] - out["cpu_mean_168"].shift(168)
    if sph > 1:                                                        # fitur yang hanya ada di data sub-jam
        c = feat[TARGET]; stepmin = 60 // sph
        out["cpu_now_sub"] = c.reindex(hourly_ix)
        for k in range(1, sph): out[f"cpu_lag_{k * stepmin}m"] = c.shift(k).reindex(hourly_ix)
        out["cpu_diff_sub"] = (c - c.shift(1)).reindex(hourly_ix)
        out["cpu_std_2jam"] = c.rolling(2 * sph, min_periods=2 * sph).std(ddof=0).reindex(hourly_ix)
        out["cpu_max_6jam"] = c.rolling(6 * sph, min_periods=6 * sph).max().reindex(hourly_ix)
        out["cpu_min_6jam"] = c.rolling(6 * sph, min_periods=6 * sph).min().reindex(hourly_ix)
    for m in tele:
        v = np.log1p(feat[m].clip(lower=0)) if m in LOG_COLS else feat[m]; vh = ke_jam(v)
        out[f"tele_{m}_now"] = vh; out[f"tele_{m}_mean_24"] = vh.rolling(24, min_periods=18).mean(); out[f"tele_{m}_lag_24"] = vh.shift(24)
        if m in TELE_LONG: r = vh.rolling(168, min_periods=150).mean(); out[f"tele_{m}_mean_168"] = r; out[f"tele_{m}_dev_168"] = vh - r
        if sph > 1: out[f"tele_{m}_diff_sub"] = (v - v.shift(1)).reindex(hourly_ix)
    s = hourly_ix - pd.Timedelta(hours=1)
    out["origin_jam"] = pd.Series(s.hour, index=hourly_ix).astype(float); out["origin_hari"] = pd.Series(s.dayofweek, index=hourly_ix).astype(float)
    out["origin_jam_sin"] = np.sin(2 * np.pi * out["origin_jam"] / 24); out["origin_jam_cos"] = np.cos(2 * np.pi * out["origin_jam"] / 24)
    out["origin_akhir_pekan"] = (out["origin_hari"] >= 5).astype(float)
    prof = profil_jam_minggu(H); hw = how(hourly_ix); pos = np.arange(len(hourly_ix))
    out["profil_jam_minggu"] = pd.Series(prof[pos, hw], index=hourly_ix)
    out["anom_now"] = H - out["profil_jam_minggu"]
    anom_hist = pd.Series(H.to_numpy() - prof[pos, hw], index=hourly_ix)                      # anomali tiap jam terhadap profil saat itu
    out["anom_24"] = anom_hist.rolling(24, min_periods=24).mean()
    X = pd.DataFrame(out, index=hourly_ix)
    return X, H, prof
def fitur_jendela(X, H, prof, cal, L):
    """Fitur khusus jendela (t, t + L]: pola musiman dan kalender di dalam jendela. Semua kausal (naif k minggu: t - 168k + L <= t)."""
    assert L <= 168
    ix = X.index; pos = np.arange(len(ix)); hw = how(ix)
    fm = H[::-1].rolling(L, min_periods=int(np.ceil(0.8 * L))).max()[::-1].shift(-1)      # maks CPU pada (t, t + L]
    out = {}
    for k in (1, 2, 3, 4): out[f"naif_max_{k}minggu"] = fm.shift(168 * k)
    if L <= 24: out["naif_max_1hari"] = fm.shift(24)                      # jendela yang sama kemarin; kausal karena t - 24 + L <= t
    nm = pd.DataFrame({k: out[f"naif_max_{k}minggu"] for k in (1, 2, 3, 4)})
    out["rata_naif_max"] = nm.mean(axis=1, skipna=True); out["maks_naif_max"] = nm.max(axis=1, skipna=True)
    idx = (hw[:, None] + np.arange(1, L + 1)[None, :]) % 168
    pw = prof[pos[:, None], idx]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        out["profil_max_jendela"] = pd.Series(np.nanmax(pw, axis=1), index=ix); out["profil_mean_jendela"] = pd.Series(np.nanmean(pw, axis=1), index=ix)
    out["dasar_max_jendela"] = out["profil_max_jendela"] + X["anom_now"] * np.exp(-1.0 / 3)
    s = ix - pd.Timedelta(hours=1); dd = s.normalize()
    libur_h = pd.Series(cal["libur"].reindex(dd).to_numpy(), index=ix); akhir_h = pd.Series((s.dayofweek >= 5).astype(float), index=ix)
    out["libur_dalam_jendela"] = libur_h[::-1].rolling(L, min_periods=1).max()[::-1].shift(-1)
    out["akhir_pekan_dalam_jendela"] = akhir_h[::-1].rolling(L, min_periods=1).mean()[::-1].shift(-1)
    out["hari_ke_libur"] = pd.Series(cal["hari_ke_libur"].reindex(dd).to_numpy(), index=ix)
    out["hari_sejak_libur"] = pd.Series(cal["hari_sejak_libur"].reindex(dd).to_numpy(), index=ix)
    out["jam_akhir_jendela"] = ((X["origin_jam"] + L) % 24)
    return pd.concat([X, pd.DataFrame(out, index=ix)], axis=1).astype(np.float32)
def label_jendela(y_jam, L):
    """y = 1 bila maks CPU per jam resmi pada (t, t + L] >= HIGH; NaN bila < 95 % jam teramati."""
    need = int(np.ceil(MIN_TARGET * L))
    fm = y_jam[::-1].rolling(L, min_periods=need).max()[::-1].shift(-1)
    return fm.ge(HIGH).astype(float).where(fm.notna())

t0 = time.time()
X0, H, prof = fitur_dasar(feat, SPH, tele)
hist_ok = H.notna().astype(int).rolling(169, min_periods=169).sum().eq(169)
y_now = y_jam.reindex(X0.index)
elig0 = (y_now.notna() & y_now.lt(HIGH) & hist_ok).fillna(False)   # C2 + syarat "dini": CPU di origin masih < HIGH
XLs, yLs, ELIGs = {}, {}, {}
for L in LEADS:
    XLs[L] = fitur_jendela(X0, H, prof, cal, L); yLs[L] = label_jendela(y_jam, L).reindex(XLs[L].index)
    ELIGs[L] = (elig0 & yLs[L].notna()).fillna(False)

# Uji kebocoran: fitur di origin t tidak berubah bila data sesudah t diacak (termasuk efek interpolasi dan fitur jendela)
L_cek = max(LEADS); uji = []
for t in [pd.Timestamp("2026-06-15 12:00"), pd.Timestamp("2026-09-10 03:00")]:
    if t not in XLs[L_cek].index: continue
    pert = df.copy(); fut = pert.index > t
    pert.loc[fut, METRICS] = np.random.default_rng(0).uniform(1, 100, size=(int(fut.sum()), len(METRICS)))
    fp_ = interpolasi_celah_pendek(pert[[TARGET] + tele], MAX_GAP_H * SPH); Xp, Hp, pp = fitur_dasar(fp_, SPH, tele)
    XLp = fitur_jendela(Xp, Hp, pp, cal, L_cek)
    a_, b_ = XLs[L_cek].loc[t].to_numpy(float), XLp.loc[t].to_numpy(float)
    ok = bool(np.allclose(np.nan_to_num(a_, nan=-999), np.nan_to_num(b_, nan=-999), atol=1e-4)); uji.append({"Origin": t, "Fitur sama sesudah data masa depan diacak": "LOLOS" if ok else "GAGAL"})
    assert ok, f"KEBOCORAN fitur di {t}"
show(pd.DataFrame(uji), caption="Tabel 2. Uji kebocoran")
show(pd.DataFrame([{"Set fitur": FS_NAME[FITUR], "Jumlah fitur": XLs[L].shape[1], "Fitur telemetri": sum(c.startswith("tele_") for c in XLs[L].columns),
                    "Fitur jendela": XLs[L].shape[1] - X0.shape[1], "Origin per jam": len(X0), "Memenuhi syarat": int(ELIGs[L].sum()), "L (jam)": L} for L in LEADS]),
     {"Jumlah fitur": "{:d}", "Fitur telemetri": "{:d}", "Fitur jendela": "{:d}", "Origin per jam": "{:,d}", "Memenuhi syarat": "{:,d}", "L (jam)": "{:d}"}, f"Tabel 3. Set fitur dan origin ({time.time() - t0:.0f} s)")

## 4. Label dan ketidakseimbangan kelas

Label *y* = 1 bila maks. CPU per jam resmi pada (t, t + L] ≥ 80 %. Tabel berikut menunjukkan seberapa langka kelas positif di tiap fase; rasio inilah yang ditangani di bagian 5–6.

In [ ]:
def origins_fase(ph, elig, a=None, b=None, L=1):
    sp = SPLIT[ph]; end = pd.Timestamp(sp["end"]); a = a or pd.Timestamp(sp["refits"][0]); b = b or end
    ix = elig.index[elig.to_numpy()]
    return ix[(ix >= a) & (ix < b) & (ix + pd.Timedelta(hours=L) <= end)]
tabel_label = []
for L in LEADS:
    for ph in SPLIT:
        o = origins_fase(ph, ELIGs[L], L=L); yo = yLs[L].loc[o]
        tabel_label.append({"L (jam)": L, "Fase": FASE[ph], "Origin": len(o), "Positif": int(yo.sum()), "Positif (%)": 100 * yo.mean(),
                            "Rasio negatif:positif": (1 - yo.mean()) / max(yo.mean(), 1e-9), "Refit": ", ".join(pd.Timestamp(r).strftime("%d %b") for r in SPLIT[ph]["refits"])})
TL = pd.DataFrame(tabel_label); TL.to_csv(f"{OUT_DIR}/tabel_label.csv", index=False)
show(TL, {"L (jam)": "{:d}", "Origin": "{:,d}", "Positif": "{:d}", "Positif (%)": "{:.1f}", "Rasio negatif:positif": "{:.1f}"}, "Tabel 4. Label per fase: kelas positif langka")

## 5. Model dan penanganan data tidak seimbang

**Model.** Regresi logistik (pembanding linear), HistGradientBoosting, LightGBM, XGBoost, dan **ensemble GBDT** (rata-rata probabilitas ketiga boosting dengan strategi yang sama; Mienye & Sun, 2022). Boosting memakai 300 iterasi, learning rate 0,05, 15 daun / kedalaman 4, sama dengan notebook forecasting. Minimal 50 sampel per daun untuk HGB dan LightGBM. Untuk XGBoost `min_child_weight` dibiarkan 1, karena parameter itu mengukur jumlah hessian p(1 − p), bukan jumlah sampel; nilai 50 di versi lama membuat XGBoost hampir tidak bisa membentuk split pada kelas langka.

**Empat strategi**, mengikuti taksonomi Rezvani & Wang (2023). Imputasi median dan resampling dihitung **sekali per refit dan strategi**, lalu dipakai bersama oleh semua keluarga model, sehingga satu-satunya perbedaan antarkeluarga adalah modelnya.

| Strategi | Tingkat | Isi |
|---|---|---|
| Tanpa penanganan | – | Pembanding. Bersama ambang yang disetel, ini sering sudah cukup dan probabilitasnya tetap terkalibrasi (van den Goorbergh dkk., 2022). |
| Bobot kelas | algoritma (cost-sensitive) | Bobot sampel positif = n_negatif / n_positif (setara `class_weight="balanced"` / `scale_pos_weight`). |
| SMOTE | data | Sampel sintetis minoritas di sepanjang garis ke salah satu 5 tetangga terdekat (Chawla dkk., 2002), **hanya pada data training**. |
| Hibrida | data + algoritma | SMOTE sampai setengah rasio target, sisanya bobot kelas (Eldeeb dkk., 2026). |

Under-sampling tidak dipakai lagi: di versi sebelumnya ia paling buruk untuk keempat keluarga model, karena membuang sekitar 95 % data mayoritas (Tarawneh dkk., 2022).

**Ambang alarm** untuk setiap konfigurasi dipilih di validation sebagai ambang yang memaksimalkan F1 (Esposito dkk., 2021). Semua metrik berambang (presisi, recall, F1, MCC, spesifisitas) dihitung pada ambang itu; PR-AUC dan ROC-AUC tidak bergantung ambang. **Brier** dilaporkan untuk melihat kalibrasi: bobot kelas dan SMOTE menggeser probabilitas ke atas, sehingga Brier-nya biasanya lebih buruk walaupun PR-AUC setara.

**Evaluasi per kejadian** (Hyland dkk., 2020): episode CPU ≥ 80 % yang mulai pada jam *o* dianggap **terdeteksi** bila ada alarm di origin [o − L, o − 1 jam]; *lead time* = jarak alarm pertama ke awal episode. Alarm palsu dilaporkan dua cara: per jam (origin dengan *y* = 0 yang dialarmkan) dan per **kejadian alarm** (alarm pada jam berturut-turut dihitung satu; palsu bila tidak satu pun jamnya diikuti kejadian). Yang kedua lebih dekat dengan beban operator.


In [ ]:
def buat_model(fam):
    if fam == "logit": return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000))
    if fam == "hgb":   return HistGradientBoostingClassifier(max_iter=N_ITER, learning_rate=LR, max_leaf_nodes=15, min_samples_leaf=50, early_stopping=False, random_state=SEED)
    if fam == "lgbm":  return lgb.LGBMClassifier(objective="binary", n_estimators=N_ITER, learning_rate=LR, num_leaves=15, min_child_samples=50, subsample=.8, subsample_freq=1,
                                                 colsample_bytree=.8, n_jobs=N_JOBS, random_state=SEED, verbose=-1)
    # XGBoost: min_child_weight = jumlah HESSIAN minimum per daun (p(1 - p) pada log-loss), bukan jumlah sampel. Nilai 50 di versi lama
    # melarang hampir semua split pada kelas langka (PR-AUC "tanpa" 0,38; "under" 0,08). Dipakai nilai bawaan 1; kedalaman tetap 4.
    if fam == "xgb":   return xgb.XGBClassifier(objective="binary:logistic", n_estimators=N_ITER, learning_rate=LR, max_depth=4, min_child_weight=1, subsample=.8,
                                                colsample_bytree=.8, tree_method="hist", n_jobs=N_JOBS, random_state=SEED, eval_metric="logloss", **XGB_DEV)
    raise ValueError(fam)
def smote(X, y, rasio, k, rng):
    """SMOTE (Chawla dkk., 2002): sampel sintetis minoritas di sepanjang garis ke salah satu k tetangga terdekat (jarak pada fitur terstandar)."""
    Xm = X[y == 1]; n_maj = int((y == 0).sum()); n_min = len(Xm)
    n_new = int(round(n_maj * rasio)) - n_min
    if n_new <= 0 or n_min < 2: return X, y, 0
    kk = min(k, n_min - 1)
    sc = StandardScaler().fit(X); Z = sc.transform(Xm)
    nn = NearestNeighbors(n_neighbors=kk + 1).fit(Z); nb = nn.kneighbors(Z, return_distance=False)[:, 1:]
    i = rng.integers(0, n_min, n_new); j = nb[i, rng.integers(0, kk, n_new)]; u = rng.random((n_new, 1))
    Xs = Xm[i] + u * (Xm[j] - Xm[i])
    return np.vstack([X, Xs]), np.concatenate([y, np.ones(n_new)]), n_new
def siapkan_training(strat, Xtr, ytr, Xte, rasio, seed):
    """Imputasi median (dari training) + penanganan ketidakseimbangan. Dihitung SEKALI per refit dan strategi, lalu dipakai semua keluarga."""
    imp = SimpleImputer(strategy="median", keep_empty_features=True).fit(Xtr)
    Xtr, Xte = imp.transform(Xtr).astype(np.float32), imp.transform(Xte).astype(np.float32)
    rng = np.random.default_rng(seed); w = None; n_ubah = 0
    if strat == "smote":     Xtr, ytr, n_ubah = smote(Xtr, ytr, rasio, K_SMOTE, rng)
    elif strat == "hibrida": Xtr, ytr, n_ubah = smote(Xtr, ytr, rasio / 2, K_SMOTE, rng)   # SMOTE hanya sampai setengah rasio target ...
    if strat in ("bobot", "hibrida"):                                                      # ... sisanya diimbangi bobot kelas (dihitung sesudah SMOTE)
        w = np.where(ytr == 1, (ytr == 0).sum() / max((ytr == 1).sum(), 1), 1.0)
    return Xtr, ytr, w, Xte, n_ubah
def fit_proba(fam, Xtr, ytr, w, Xte):
    m = buat_model(fam)
    if fam == "logit": m.fit(Xtr, ytr, logisticregression__sample_weight=w)
    else: m.fit(Xtr, ytr, sample_weight=w)
    return m.predict_proba(Xte)[:, 1]
def ambang_f1(y, p):
    """Ambang yang memaksimalkan F1 di validation (threshold moving; Esposito dkk., 2021)."""
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
    return float(thr[int(np.argmax(f1))])
def metrik(y, p, thr):
    y = np.asarray(y, int); p = np.asarray(p, float); yhat = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    prec = tp / (tp + fp) if tp + fp else 0.0; rec = tp / (tp + fn) if tp + fn else 0.0; spec = tn / (tn + fp) if tn + fp else 0.0
    dua = len(np.unique(y)) == 2
    return {"n": int(len(y)), "positif": int(y.sum()), "PR-AUC": average_precision_score(y, p) if dua else np.nan,
            "ROC-AUC": roc_auc_score(y, p) if dua else np.nan, "Brier": brier_score_loss(y, np.clip(p, 0, 1)) if np.nanmax(p) <= 1 else np.nan,
            "Ambang": thr, "Presisi": prec, "Recall": rec, "Spesifisitas": spec, "F1": 2 * prec * rec / (prec + rec) if prec + rec else 0.0,
            "MCC": matthews_corrcoef(y, yhat) if dua else np.nan, "Akurasi berimbang": (rec + spec) / 2, "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn)}
def metrik_episode(y_jam, alarm, L, a, b):
    """Evaluasi per kejadian: episode CPU >= HIGH yang mulai di [a, b). Terdeteksi bila ada alarm di origin [awal - L, awal - 1 jam]."""
    hi = y_jam.ge(HIGH); onset = hi & ~hi.shift(1, fill_value=False) & y_jam.shift(1).notna()
    onsets = [t for t in y_jam.index[onset] if a <= t < b]
    rows = []
    for t in onsets:
        w = alarm.loc[t - pd.Timedelta(hours=L): t - pd.Timedelta(hours=1)]
        if len(w) == 0: continue                                        # tidak ada origin yang dinilai di jendela itu
        fired = w[w]; rows.append({"awal_episode": t, "terdeteksi": bool(len(fired)), "lead_jam": (t - fired.index[0]).total_seconds() / 3600 if len(fired) else np.nan})
    ep = pd.DataFrame(rows, columns=["awal_episode", "terdeteksi", "lead_jam"])
    return ep
def kejadian_alarm_palsu(origin, alarm, y):
    """Alarm pada jam berturut-turut = satu kejadian alarm; palsu bila tidak satu pun jamnya diikuti kejadian (semua y = 0)."""
    d = pd.DataFrame({"t": pd.DatetimeIndex(origin), "a": np.asarray(alarm, bool), "y": np.asarray(y, float)}).sort_values("t")
    d = d[d.a]
    if d.empty: return 0, 0
    g = d.groupby((d.t.diff() != pd.Timedelta(hours=1)).cumsum()).y.max()
    return int(len(g)), int((g == 0).sum())
def blok_bootstrap(origins, n_boot=N_BOOT, blok=BLOK_HARI, seed=SEED):
    """Indeks resample block bootstrap: blok hari berurutan diambil DENGAN pengembalian, menjaga autokorelasi antarjam."""
    days = pd.DatetimeIndex(origins).normalize(); ud = days.unique(); nb = int(np.ceil(len(ud) / blok)); rng = np.random.default_rng(seed)
    pos = [np.flatnonzero(days == d) for d in ud]
    return [np.concatenate([pos[j] for s0 in rng.integers(0, max(len(ud) - blok, 0) + 1, nb) for j in range(s0, min(s0 + blok, len(ud)))]) for _ in range(n_boot)]


## 6. Validation: semua konfigurasi, lalu seleksi dibekukan

Grid = 5 keluarga × 4 strategi = 20 konfigurasi per jendela. Yang benar-benar dilatih hanya 4 keluarga × 4 strategi (ensemble = rata-rata probabilitas tiga boosting), walk-forward dengan refit bulanan; label training harus sudah matang sebelum refit.

**Aturan seleksi** (hanya validation), per jendela:
1. Hitung PR-AUC setiap konfigurasi dan galat bakunya (SE) dengan block bootstrap blok 7 hari (500 ulangan, resample yang sama untuk semua konfigurasi).
2. Konfigurasi "setara" = PR-AUC ≥ PR-AUC tertinggi − 1 SE konfigurasi tertinggi (**aturan satu galat baku**; James dkk., 2021, §6.1.3).
3. Dari yang setara, pilih yang paling sederhana: strategi tanpa < bobot < SMOTE < hibrida, lalu keluarga logistik < boosting tunggal < ensemble, lalu PR-AUC tertinggi.

Alasannya: dengan ±140 positif di validation, selisih PR-AUC beberapa perseribu antara konfigurasi teratas hanyalah derau; memilih yang tertinggi mentah-mentah cenderung memilih konfigurasi yang kebetulan cocok dengan validation dan lebih rumit tanpa manfaat nyata.

Ambang pembanding (ambang statis CPU dan naif musiman) juga disetel di validation. Seleksi ditulis ke `seleksi_beku_ews.json` dengan hash; sel ini menolak menimpanya bila test sudah dijalankan.


In [ ]:
GRID = [{"fam": f, "strat": s} for f in FAM for s in STRAT]
def key(c): return f"{c['fam']}|{c['strat']}"
def label(c): return f"{FAM[c['fam']]} · {STRAT[c['strat']]}"
def latih_dari(specs):
    """Pasangan (keluarga, strategi) yang benar-benar dilatih; ensemble butuh ketiga GBDT dengan strategi yang sama."""
    return sorted({(f, c["strat"]) for c in specs for f in (GBDT if c["fam"] == "ens" else (c["fam"],))}, key=str)
def path_cache(ph, L, specs):
    k = sha_obj([ph, L, latih_dari(specs), list(XLs[L].columns), [str(t) for t in ELIGs[L].index[ELIGs[L]]], N_ITER, LR, RASIO, VERSIONS, DATA_SHA, LABEL_SHA, PIPELINE_VERSION])[:12]
    return f"{OUT_DIR}/cache/{ph}_L{L}_{k}.pkl"

def run_phase(ph, L, specs):
    XL, yL, elig = XLs[L], yLs[L], ELIGs[L]; latih = latih_dari(specs); path = path_cache(ph, L, specs)
    if REUSE_CACHE and os.path.exists(path): return pickle.load(open(path, "rb")), "cache"
    t1 = time.time(); parts, log = [], []
    sp = SPLIT[ph]; refits = [pd.Timestamp(r) for r in sp["refits"]] + [pd.Timestamp(sp["end"])]
    for a, b in zip(refits[:-1], refits[1:]):
        so = origins_fase(ph, elig, a, b, L)
        tr_ix = elig.index[elig & (elig.index + pd.Timedelta(hours=L) <= a)]          # label training sudah matang sebelum refit
        Xtr, ytr = XL.loc[tr_ix].to_numpy(np.float32), yL.loc[tr_ix].to_numpy(float)
        Xte = XL.loc[so].to_numpy(np.float32)
        base = pd.DataFrame({"origin": so, "L": L, "phase": ph, "refit": a, "y": yL.loc[so].to_numpy(float), "cpu_now": y_jam.reindex(so).to_numpy()})
        base["naif"] = XL.loc[so, "naif_max_1minggu"].to_numpy(float)
        for s in dict.fromkeys(st for _, st in latih):
            Xs, ys, w, Xt, n_ubah = siapkan_training(s, Xtr, ytr, Xte, RASIO, SEED)      # sekali per refit dan strategi
            for f in [f for f, st in latih if st == s]:
                t2 = time.time(); base[f"{f}|{s}"] = fit_proba(f, Xs, ys, w, Xt)
                log.append({"fase": ph, "L": L, "refit": str(a.date()), "model": f"{FAM[f]} · {STRAT[s]}", "baris_training_asli": len(ytr), "positif_asli": int(ytr.sum()),
                            "baris_setelah_resampling": len(ys), "positif_setelah": int(ys.sum()), "baris_diubah": n_ubah, "detik": round(time.time() - t2, 1)})
            if all(f"{f}|{s}" in base for f in GBDT): base[f"ens|{s}"] = base[[f"{f}|{s}" for f in GBDT]].mean(axis=1)
        parts.append(base)
    out = (pd.concat(parts, ignore_index=True), pd.DataFrame(log)); pickle.dump(out, open(path, "wb"))
    return out, f"dilatih {(time.time() - t1) / 60:.1f} menit"

SEL_PATH = f"{OUT_DIR}/seleksi_beku_ews.json"; FLAG = f"{OUT_DIR}/test_ews_sudah_dijalankan.flag"
PV, LOGV, GRIDRES, SEL = {}, [], [], {}
for L in LEADS:
    (pv, logv), how_v = run_phase("validation", L, GRID); PV[L] = pv; LOGV.append(logv)
    print(f"validation L={L:>2}: {how_v} | {len(GRID)} konfigurasi | {pv.origin.nunique()} origin | positif {pv.y.mean() * 100:.1f} %")
    rows = []
    for c in GRID:
        thr = ambang_f1(pv.y, pv[key(c)]); m = metrik(pv.y, pv[key(c)], thr)
        rows.append({"L (jam)": L, "Keluarga": FAM[c["fam"]], "Strategi": STRAT[c["strat"]], "kunci": key(c), **m})
    g = pd.DataFrame(rows)
    # Aturan satu galat baku (James dkk., 2021, §6.1.3); SE PR-AUC dari block bootstrap dengan resample yang sama untuk semua konfigurasi
    yv = pv.y.to_numpy(); P_ = np.column_stack([pv[key(c)].to_numpy() for c in GRID])
    boot = np.array([[average_precision_score(yv[i], P_[i, j]) for j in range(len(GRID))] for i in blok_bootstrap(pv.origin) if 0 < yv[i].sum() < len(i)])
    g["SE PR-AUC"] = boot.std(axis=0, ddof=1)
    ib = int(g["PR-AUC"].idxmax()); batas = float(g.loc[ib, "PR-AUC"] - g.loc[ib, "SE PR-AUC"])
    g["Dalam 1 SE"] = g["PR-AUC"] >= batas
    g["_s"] = [list(STRAT).index(c["strat"]) for c in GRID]; g["_f"] = [FAM_URUT[c["fam"]] for c in GRID]
    best = g[g["Dalam 1 SE"]].sort_values(["_s", "_f", "PR-AUC"], ascending=[True, True, False]).iloc[0]
    g = g.drop(columns=["_s", "_f"]); GRIDRES.append(g)
    SEL[L] = {"kunci": best["kunci"], "model": f"{best['Keluarga']} · {best['Strategi']}", "ambang": float(best["Ambang"]),
              "pr_auc_validation": float(best["PR-AUC"]), "mcc_validation": float(best["MCC"]), "f1_validation": float(best["F1"]), "brier_validation": float(best["Brier"]),
              "pr_auc_tertinggi": float(g.loc[ib, "PR-AUC"]), "konfigurasi_tertinggi": g.loc[ib, "kunci"], "se_tertinggi": float(g.loc[ib, "SE PR-AUC"]), "batas_1se": batas,
              "ulangan_bootstrap_sah": int(len(boot))}
    SEL[L]["ambang_pembanding"] = {"ambang_cpu": ambang_f1(pv.y, pv.cpu_now), "naif": ambang_f1(pv.y, pv.naif.fillna(0))}
GR = pd.concat(GRIDRES, ignore_index=True); GR.to_csv(f"{OUT_DIR}/metrik_validation_grid.csv", index=False)
LOGV = pd.concat(LOGV, ignore_index=True); LOGV.to_csv(f"{OUT_DIR}/log_pelatihan_validation.csv", index=False)

sel = {"dibekukan": time.strftime("%Y-%m-%d %H:%M:%S"), "versi_pipeline": PIPELINE_VERSION, "resolusi_menit": RES, "set_fitur": FS_NAME[FITUR], "jendela_jam": LEADS, "ambang_cpu_tinggi": HIGH,
       "aturan": ("per L: aturan satu galat baku. Konfigurasi setara = PR-AUC validation >= PR-AUC tertinggi - 1 SE (block bootstrap 7 hari, 500 ulangan). "
                  "Dari yang setara dipilih yang paling sederhana: strategi tanpa < bobot < SMOTE < hibrida, lalu keluarga logistik < boosting tunggal < ensemble, "
                  "lalu PR-AUC tertinggi. Ambang alarm = argmax F1 di validation."),
       "strategi_ketidakseimbangan": STRAT, "keluarga": FAM, "rasio_sampling": RASIO, "k_smote": K_SMOTE, "bootstrap": {"n": N_BOOT, "blok_hari": BLOK_HARI},
       "seleksi": {str(L): SEL[L] for L in LEADS}, "telemetri_terpilih": tele, "fitur": {str(L): list(XLs[L].columns) for L in LEADS},
       "boosting": {"n_iter": N_ITER, "learning_rate": LR, "xgb_min_child_weight": 1}, "pembersihan": {"max_gap_jam": MAX_GAP_H, "min_target": MIN_TARGET}, "riwayat_panjang": RIWAYAT_PANJANG,
       "data_sha256": DATA_SHA, "label_sha256": LABEL_SHA, "data_cutoff": DATA_CUTOFF, "versi_pustaka": VERSIONS}
sel = json.loads(json.dumps(sel, default=str))
if os.path.exists(SEL_PATH):
    lama = json.load(open(SEL_PATH))
    assert sha_file(SEL_PATH) == open(SEL_PATH + ".sha256").read().strip(), "seleksi beku berubah setelah dibekukan"
    if {k: v for k, v in lama.items() if k != "dibekukan"} != {k: v for k, v in sel.items() if k != "dibekukan"}:
        assert not (os.path.exists(FLAG) and not os.environ.get("FORCE_RERUN_TEST")), "Seleksi validation berubah padahal test sudah dijalankan. Hapus folder output atau set FORCE_RERUN_TEST=1 bila memang ingin mengulang (eksploratori)."
        json.dump(sel, open(SEL_PATH, "w"), indent=1); open(SEL_PATH + ".sha256", "w").write(sha_file(SEL_PATH))
else:
    json.dump(sel, open(SEL_PATH, "w"), indent=1); open(SEL_PATH + ".sha256", "w").write(sha_file(SEL_PATH))
print("seleksi beku sha256:", open(SEL_PATH + ".sha256").read()[:12])


In [ ]:
# Tabel 5: grid lengkap validation (PR-AUC, ROC-AUC, presisi, recall, F1, MCC) per keluarga x strategi
FMT = {"L (jam)": "{:d}", **{c: "{:.3f}" for c in KOLOM_METRIK}, "SE PR-AUC": "{:.3f}", "Ambang": "{:.3f}", "TP": "{:d}", "FP": "{:d}", "FN": "{:d}"}
show(GR.sort_values(["L (jam)", "PR-AUC"], ascending=[True, False])[["L (jam)", "Keluarga", "Strategi", "PR-AUC", "SE PR-AUC", "Dalam 1 SE"] + KOLOM_METRIK[1:] + ["Ambang", "TP", "FP", "FN"]], FMT,
     "Tabel 5. Validation: semua konfigurasi, diurutkan menurut PR-AUC (metrik berambang pada ambang maks. F1; \"Dalam 1 SE\" = setara dengan yang tertinggi)")
ST = GR.groupby(["L (jam)", "Strategi"], sort=False)[["PR-AUC", "Presisi", "Recall", "F1", "MCC", "Brier"]].mean().reset_index()
ST.to_csv(f"{OUT_DIR}/ringkasan_strategi_validation.csv", index=False)
show(ST, FMT, "Tabel 6. Rata-rata metrik validation per strategi ketidakseimbangan (rata-rata atas 5 keluarga model; Brier makin kecil makin terkalibrasi)")
SK = GR.groupby(["L (jam)", "Keluarga"], sort=False)[["PR-AUC", "Presisi", "Recall", "F1", "MCC", "Brier"]].mean().reset_index()
show(SK, FMT, "Tabel 7. Rata-rata metrik validation per keluarga model (rata-rata atas 4 strategi)")
show(pd.DataFrame([{"L (jam)": L, "Model terpilih": SEL[L]["model"], "PR-AUC validation": SEL[L]["pr_auc_validation"], "PR-AUC tertinggi": SEL[L]["pr_auc_tertinggi"],
                    "SE": SEL[L]["se_tertinggi"], "Batas 1 SE": SEL[L]["batas_1se"], "MCC validation": SEL[L]["mcc_validation"], "F1 validation": SEL[L]["f1_validation"],
                    "Brier validation": SEL[L]["brier_validation"], "Ambang alarm": SEL[L]["ambang"]} for L in LEADS]), {"L (jam)": "{:d}"},
     "Tabel 8. Seleksi beku per jendela (aturan satu galat baku)")

In [ ]:
# Gambar 1: PR-AUC validation per keluarga x strategi
nL = len(LEADS)
fig, axs = plt.subplots(1, nL, figsize=(3.6 * nL + 1, 3.3), sharey=True)
for ax, L in zip(np.atleast_1d(axs), LEADS):
    g = GR[GR["L (jam)"] == L]; x = np.arange(len(STRAT)); wbar = 0.16
    for i, (fk, fn) in enumerate(FAM.items()):
        v = [g[(g.Keluarga == fn) & (g.Strategi == STRAT[s])]["PR-AUC"].iloc[0] for s in STRAT]
        ax.bar(x + (i - (len(FAM) - 1) / 2) * wbar, v, width=wbar * 0.9, color=COL[fk], label=fn)
    ax.axhline(g["positif"].iloc[0] / g["n"].iloc[0], color=COL["base"], ls=":", lw=.9, label="Acak (prevalensi)")
    ax.set_xticks(x); ax.set_xticklabels([STRAT_PENDEK[k] for k in STRAT]); ax.set_title(f"L = {L} jam"); ax.set_ylim(0, 1)
np.atleast_1d(axs)[0].set_ylabel("PR-AUC validation")
hd, lb = np.atleast_1d(axs)[0].get_legend_handles_labels(); fig.legend(hd, lb, fontsize=7.5, ncol=6, loc="lower center", bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f"Gambar 1. PR-AUC validation per keluarga dan strategi ketidakseimbangan · data {RES_NAME[RES]}", x=0.01, ha="left", fontsize=10)
fig.tight_layout(rect=(0, 0.06, 1, 1)); fig.savefig(f"{OUT_DIR}/figures/gambar1_grid_validation.png", bbox_inches="tight"); plt.show()

## 7. Test (sekali, dengan seleksi beku)

Hanya konfigurasi terpilih yang dijalankan. Bila flag test sudah ada, hasil dibaca dari cache; menjalankan ulang tanpa cache butuh `FORCE_RERUN_TEST=1`.

In [ ]:
PT = {}
for L in LEADS:
    spec = next(c for c in GRID if key(c) == SEL[L]["kunci"])
    assert not (os.path.exists(FLAG) and not os.path.exists(path_cache("test", L, [spec])) and not os.environ.get("FORCE_RERUN_TEST")), \
        "Test sudah pernah dijalankan tetapi cache-nya tidak ada. Set FORCE_RERUN_TEST=1 bila memang ingin mengulang."
    (pt, logt), how_t = run_phase("test", L, [spec]); PT[L] = pt
    logt.to_csv(f"{OUT_DIR}/log_pelatihan_test_L{L}.csv", index=False); print(f"test L={L:>2}: {how_t} | {pt.origin.nunique()} origin | positif {pt.y.mean() * 100:.1f} %")
if not os.path.exists(FLAG): open(FLAG, "w").write(f"test dijalankan {time.strftime('%Y-%m-%d %H:%M:%S')} dengan seleksi sha256 {open(SEL_PATH + '.sha256').read()[:12]}")
print(open(FLAG).read())

## 8. Evaluasi

**Per origin:** PR-AUC, ROC-AUC, presisi, recall, F1, MCC, spesifisitas pada ambang beku, dan Brier. Pembanding: ambang statis CPU saat ini (sistem peringatan konvensional) dan naif musiman (maks. CPU pada jendela yang sama minggu lalu), dengan ambang yang juga disetel di validation.

**Ketidakpastian:** CI 95 % block bootstrap (blok 7 hari, 500 ulangan) untuk PR-AUC model terpilih dan selisihnya terhadap ambang statis. Test hanya punya sekitar 30 positif, jadi angka tunggal tanpa CI mudah menyesatkan.

**Per kejadian:** persentase episode CPU ≥ 80 % yang terdeteksi, lead time alarm pertama, alarm palsu per jam, dan kejadian alarm palsu per minggu.


In [ ]:
rows, rows_ep, rows_ci, pred_parts = [], [], [], []
for L in LEADS:
    kb = SEL[L]["kunci"]; thr = SEL[L]["ambang"]; ap = SEL[L]["ambang_pembanding"]
    for ph, P in (("validation", PV[L]), ("test", PT[L])):
        P = P.sort_values("origin").reset_index(drop=True); P["prob_terpilih"] = P[kb]; P["alarm"] = P[kb] >= thr
        skor = {SEL[L]["model"]: (P[kb], thr), BASEL["ambang_cpu"]: (P.cpu_now, ap["ambang_cpu"]), BASEL["naif"]: (P.naif.fillna(0), ap["naif"])}
        for nama, (s, t_) in skor.items(): rows.append({"L (jam)": L, "Fase": FASE[ph], "Model": nama, **metrik(P.y, s, t_)})
        # CI 95 % block bootstrap: PR-AUC model terpilih dan selisihnya terhadap ambang statis CPU
        yv, pm, pc = P.y.to_numpy(), P[kb].to_numpy(), P.cpu_now.to_numpy()
        v = np.array([(average_precision_score(yv[i], pm[i]), average_precision_score(yv[i], pm[i]) - average_precision_score(yv[i], pc[i]))
                      for i in blok_bootstrap(P.origin) if 0 < yv[i].sum() < len(i)])
        titik = (average_precision_score(yv, pm), average_precision_score(yv, pm) - average_precision_score(yv, pc))
        for j, nm in enumerate(("PR-AUC model terpilih", "Selisih PR-AUC: model − ambang statis CPU")):
            lo, hi = np.percentile(v[:, j], [2.5, 97.5])
            rows_ci.append({"L (jam)": L, "Fase": FASE[ph], "Besaran": nm, "Nilai": titik[j], "CI 95% bawah": lo, "CI 95% atas": hi, "Ulangan sah": len(v),
                            "Kesimpulan": "" if j == 0 else ("model lebih baik" if lo > 0 else "model lebih buruk" if hi < 0 else "tidak berbeda nyata")})
        a, b = pd.Timestamp(SPLIT[ph]["refits"][0]), pd.Timestamp(SPLIT[ph]["end"]); minggu = (b - a).days / 7
        for nama, (s, t_) in skor.items():
            al = (s >= t_).to_numpy(); ep = metrik_episode(y_jam, pd.Series(al, index=pd.DatetimeIndex(P.origin)), L, a, b)
            n_kej, n_palsu = kejadian_alarm_palsu(P.origin, al, P.y)
            rows_ep.append({"L (jam)": L, "Fase": FASE[ph], "Model": nama, "Episode": len(ep), "Terdeteksi": int(ep.terdeteksi.sum()),
                            "Deteksi (%)": 100 * ep.terdeteksi.mean() if len(ep) else np.nan, "Lead time rata-rata (jam)": ep.lead_jam.mean(),
                            "Lead time median (jam)": ep.lead_jam.median(), "Alarm palsu (jam)": int(((P.y == 0).to_numpy() & al).sum()),
                            "Kejadian alarm": n_kej, "Kejadian alarm palsu": n_palsu, "Kejadian alarm palsu per minggu": n_palsu / minggu})
        pred_parts.append(P[["origin", "L", "phase", "refit", "y", "cpu_now", "naif", "prob_terpilih", "alarm"] + [c for c in P.columns if "|" in c]])
MT = pd.DataFrame(rows); MT.to_csv(f"{OUT_DIR}/metrik_ews.csv", index=False)
EP = pd.DataFrame(rows_ep); EP.to_csv(f"{OUT_DIR}/metrik_episode.csv", index=False)
CI = pd.DataFrame(rows_ci); CI.to_csv(f"{OUT_DIR}/ci_bootstrap.csv", index=False)
PRED = pd.concat(pred_parts, ignore_index=True); PRED.to_csv(f"{OUT_DIR}/prediksi_ews.csv.gz", index=False, compression="gzip")
show(MT[["L (jam)", "Fase", "Model", "n", "positif"] + KOLOM_METRIK + ["FP", "FN"]], {**FMT, "n": "{:d}", "positif": "{:d}"},
     "Tabel 9. Model terpilih dan pembanding: PR-AUC, ROC-AUC, presisi, recall, F1, MCC, spesifisitas, Brier (validation dan test)")
show(CI, {"L (jam)": "{:d}", "Nilai": "{:+.3f}", "CI 95% bawah": "{:+.3f}", "CI 95% atas": "{:+.3f}", "Ulangan sah": "{:d}"},
     "Tabel 9b. CI 95 % block bootstrap (blok 7 hari) untuk PR-AUC model terpilih dan selisihnya terhadap ambang statis CPU")
show(EP, {"L (jam)": "{:d}", "Episode": "{:d}", "Terdeteksi": "{:d}", "Deteksi (%)": "{:.1f}", "Lead time rata-rata (jam)": "{:.2f}", "Lead time median (jam)": "{:.1f}",
          "Alarm palsu (jam)": "{:d}", "Kejadian alarm": "{:d}", "Kejadian alarm palsu": "{:d}", "Kejadian alarm palsu per minggu": "{:.1f}"},
     "Tabel 10. Evaluasi per kejadian: episode CPU ≥ 80 % yang terdeteksi, lead time, dan alarm palsu (per jam dan per kejadian alarm)")


In [ ]:
# Gambar 2: kurva precision-recall, model terpilih vs pembanding
fig, axs = plt.subplots(2, nL, figsize=(3.4 * nL + 1, 5.8), sharex=True, sharey=True); axs = np.array(axs).reshape(2, nL)
for j, L in enumerate(LEADS):
    for i, ph in enumerate(("validation", "test")):
        ax = axs[i, j]; P = PRED[(PRED.L == L) & (PRED.phase == ph)]
        if P.y.nunique() < 2: continue
        pr, rc, _ = precision_recall_curve(P.y, P.prob_terpilih); ax.plot(rc, pr, color=COL["hgb"], lw=1.8, label="Model terpilih")
        m = metrik(P.y, P.prob_terpilih, SEL[L]["ambang"]); ax.plot(m["Recall"], m["Presisi"], "o", color=COL["hgb"], ms=6, mec="white")
        pr, rc, _ = precision_recall_curve(P.y, P.cpu_now); ax.plot(rc, pr, color=COL["base"], lw=1.2, ls="--", label="Ambang statis CPU")
        pr, rc, _ = precision_recall_curve(P.y, P.naif.fillna(0)); ax.plot(rc, pr, color=COL["base"], lw=1.2, ls=":", label="Naif musiman")
        ax.axhline(P.y.mean(), color=GRIDC, lw=.8); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
        if i == 0: ax.set_title(f"L = {L} jam")
        if j == 0: ax.set_ylabel(f"{FASE[ph]}\npresisi")
        if i == 1: ax.set_xlabel("recall")
hd, lb = axs[0, 0].get_legend_handles_labels(); fig.legend(hd, lb, fontsize=7.5, ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f"Gambar 2. Kurva precision-recall model terpilih dan pembanding (titik = ambang terpilih) · data {RES_NAME[RES]}", x=0.01, ha="left", fontsize=10)
fig.tight_layout(rect=(0, 0.05, 1, 1)); fig.savefig(f"{OUT_DIR}/figures/gambar2_pr_curve.png", bbox_inches="tight"); plt.show()

In [ ]:
# Gambar 3: linimasa test, alarm vs kejadian (jendela pertama)
Lg = LEADS[0]; P = PRED[(PRED.L == Lg) & (PRED.phase == "test")].set_index("origin").sort_index()
a, b = pd.Timestamp(SPLIT["test"]["refits"][0]), pd.Timestamp(SPLIT["test"]["end"]); yt = y_jam[a:b]
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.fill_between(yt.index, 0, 100, where=yt.ge(HIGH).to_numpy(), color=BAND, step="mid", label="CPU ≥ 80 % (kejadian)")
ax.plot(yt.index, yt, color=INK, lw=.7, label="CPU per jam"); ax.axhline(HIGH, color=COL["base"], lw=.8, ls="--")
tp_ = P[P.alarm & (P.y == 1)]; fp_ = P[P.alarm & (P.y == 0)]
ax.plot(tp_.index, np.full(len(tp_), 104), "|", color=COL["hgb"], ms=9, mew=1.2, label=f"Alarm benar (L = {Lg} jam)")
ax.plot(fp_.index, np.full(len(fp_), 108), "|", color=COL["lgbm"], ms=9, mew=1.2, label="Alarm palsu")
ax.set_ylim(0, 112); ax.set_ylabel("CPU (%)"); ax.legend(fontsize=7.5, ncol=4, loc="lower left"); ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.set_title(f"Gambar 3. Alarm model terpilih pada periode test · data {RES_NAME[RES]}")
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/figures/gambar3_linimasa_test.png", bbox_inches="tight"); plt.show()

# Gambar 4: sebaran lead time per jendela (model terpilih, validation + test)
fig, axs = plt.subplots(1, nL, figsize=(3.4 * nL + 1, 3.0), sharey=True)
for ax, L in zip(np.atleast_1d(axs), LEADS):
    leads = []
    for ph in SPLIT:
        Pp = PRED[(PRED.L == L) & (PRED.phase == ph)]; alarm = pd.Series(Pp.alarm.to_numpy(), index=pd.DatetimeIndex(Pp.origin)).sort_index()
        ep = metrik_episode(y_jam, alarm, L, pd.Timestamp(SPLIT[ph]["refits"][0]), pd.Timestamp(SPLIT[ph]["end"])); leads += ep.lead_jam.dropna().tolist()
    ax.hist(leads, bins=np.arange(0.5, L + 1.5, 1), color=COL["hgb"], rwidth=.85); ax.set_title(f"L = {L} jam (n = {len(leads)} episode)"); ax.set_xlabel("lead time (jam)")
np.atleast_1d(axs)[0].set_ylabel("jumlah episode")
fig.suptitle(f"Gambar 4. Lead time alarm pertama sebelum episode CPU ≥ 80 % · data {RES_NAME[RES]}", x=0.01, ha="left", fontsize=10)
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/figures/gambar4_lead_time.png", bbox_inches="tight"); plt.show()

In [ ]:
# Ringkasan markdown (untuk dikutip di skripsi)
md = [f"# Ringkasan EWS · data {RES_NAME[RES]} ({time.strftime('%Y-%m-%d %H:%M')})", "",
      f"Set fitur: {FS_NAME[FITUR]}. Jendela: {LEADS} jam. Rasio sampling SMOTE: {RASIO}. Pipeline v{PIPELINE_VERSION}. Device: {VERSIONS['device']}.", "",
      "## Label per fase", md_table(TL, ".1f"), "",
      "## Konfigurasi terpilih per jendela (aturan satu galat baku)",
      md_table(pd.DataFrame([{"L (jam)": L, "Model terpilih": SEL[L]["model"], "PR-AUC val": SEL[L]["pr_auc_validation"], "PR-AUC tertinggi": SEL[L]["pr_auc_tertinggi"],
                              "Batas 1 SE": SEL[L]["batas_1se"], "MCC val": SEL[L]["mcc_validation"], "Ambang": SEL[L]["ambang"]} for L in LEADS])), "",
      "## Grid validation lengkap", md_table(GR.sort_values(["L (jam)", "PR-AUC"], ascending=[True, False])[["L (jam)", "Keluarga", "Strategi", "SE PR-AUC", "Dalam 1 SE"] + KOLOM_METRIK]), "",
      "## Rata-rata metrik validation per strategi ketidakseimbangan", md_table(ST), "",
      "## Metrik model terpilih dan pembanding", md_table(MT[["L (jam)", "Fase", "Model", "n", "positif"] + KOLOM_METRIK + ["FP", "FN"]]), "",
      "## CI 95 % block bootstrap", md_table(CI), "",
      "## Evaluasi per kejadian (episode CPU ≥ 80 %)", md_table(EP, ".2f"), "",
      "Catatan: seleksi hanya dari validation; test dijalankan sekali dengan seleksi beku. Periode test sudah pernah dilihat di notebook forecasting, jadi hasil test eksploratori."]
open(f"{OUT_DIR}/ringkasan_ews.md", "w").write("\n".join(md)); print(f"ditulis {OUT_DIR}/ringkasan_ews.md")

## 9. Perbandingan tiga resolusi

Dijalankan bila `outputs_ews_60/`, `outputs_ews_30/`, dan `outputs_ews_15/` sudah ada (minimal dua). Model terpilih tiap resolusi dinilai pada **origin dan label yang sama** (irisan origin yang memenuhi syarat di semua resolusi). Selisih terhadap 1 jam diuji dengan *block bootstrap* blok 7 hari (500 ulangan, blok diambil dengan pengembalian); kesimpulan "lebih baik" / "lebih buruk" hanya bila seluruh CI 95 % berada di satu sisi nol. Tabel 14 menyandingkan evaluasi per kejadian ketiga resolusi.

In [ ]:
OUT_BANDING = "outputs_ews_banding"; os.makedirs(f"{OUT_BANDING}/figures", exist_ok=True)
D = {}; SELS = {}
for r in (60, 30, 15):
    p = f"outputs_ews_{r}/prediksi_ews.csv.gz"; s_ = f"outputs_ews_{r}/seleksi_beku_ews.json"
    if not os.path.exists(p): print(f"lewati {RES_NAME[r]}: {p} belum ada"); continue
    assert sha_file(s_) == open(s_ + ".sha256").read().strip(), f"seleksi beku {r} berubah"
    D[r] = pd.read_csv(p, parse_dates=["origin"]); SELS[r] = json.load(open(s_))
if len(D) >= 2:
    LEADS_B = sorted(set.intersection(*[set(d.L.unique()) for d in D.values()])); rows, boot = [], []
    for L in LEADS_B:
        for ph in ("validation", "test"):
            parts = {r: d[(d.L == L) & (d.phase == ph)].set_index("origin")[["y", "prob_terpilih", "alarm"]] for r, d in D.items()}
            common = sorted(set.intersection(*[set(p.index) for p in parts.values()]))
            if len(common) < 20: continue
            parts = {r: p.loc[common] for r, p in parts.items()}; y = next(iter(parts.values())).y.to_numpy()
            for r, p in parts.items():
                assert np.allclose(p.y.to_numpy(), y), "label berbeda antarresolusi pada origin yang sama"
                m = metrik(y, p.prob_terpilih, SELS[r]["seleksi"][str(L)]["ambang"])
                rows.append({"L (jam)": L, "Fase": FASE[ph], "Resolusi": RES_NAME[r], "Model terpilih": SELS[r]["seleksi"][str(L)]["model"], **m})
            if 60 not in parts: continue
            stat = {r: [] for r in parts if r != 60}
            for idx in blok_bootstrap(common):                                  # blok 7 hari dengan pengembalian
                if len(np.unique(y[idx])) < 2: continue
                m60 = metrik(y[idx], parts[60].prob_terpilih.to_numpy()[idx], SELS[60]["seleksi"][str(L)]["ambang"])
                for r in stat:
                    mr = metrik(y[idx], parts[r].prob_terpilih.to_numpy()[idx], SELS[r]["seleksi"][str(L)]["ambang"])
                    stat[r].append((mr["PR-AUC"] - m60["PR-AUC"], mr["MCC"] - m60["MCC"], mr["F1"] - m60["F1"]))
            for r, v in stat.items():
                v = np.array(v)
                if len(v) < 50: continue
                for j, nm in enumerate(("PR-AUC", "MCC", "F1")):
                    lo, hi = np.percentile(v[:, j], [2.5, 97.5])
                    boot.append({"L (jam)": L, "Fase": FASE[ph], "Perbandingan": f"{RES_NAME[r]} − 1 jam", "Metrik": nm, "Selisih": float(np.mean(v[:, j])), "CI 95% bawah": lo, "CI 95% atas": hi,
                                 "Kesimpulan": "lebih baik" if lo > 0 else ("lebih buruk" if hi < 0 else "tidak berbeda nyata")})
    MTB = pd.DataFrame(rows); MTB.to_csv(f"{OUT_BANDING}/metrik_tiga_resolusi.csv", index=False)
    BT = pd.DataFrame(boot); BT.to_csv(f"{OUT_BANDING}/bootstrap_selisih_vs_1jam.csv", index=False)
    show(MTB[["L (jam)", "Fase", "Resolusi", "Model terpilih", "n", "positif"] + KOLOM_METRIK], {**FMT, "n": "{:d}", "positif": "{:d}"},
         "Tabel 11. Model EWS terpilih per resolusi pada origin yang sama: PR-AUC, ROC-AUC, presisi, recall, F1, MCC")
    show(BT, {"L (jam)": "{:d}", "Selisih": "{:+.3f}", "CI 95% bawah": "{:+.3f}", "CI 95% atas": "{:+.3f}"}, "Tabel 12. Selisih terhadap 1 jam (block bootstrap blok 7 hari, 500 ulangan); positif = resolusi itu lebih baik")
    best = MTB[MTB.Fase == "Validation"].sort_values(["L (jam)", "PR-AUC"], ascending=[True, False]).groupby("L (jam)").head(1)
    show(best[["L (jam)", "Resolusi", "Model terpilih", "PR-AUC", "F1", "MCC"]], FMT, "Tabel 13. Resolusi terbaik per jendela (PR-AUC validation)")
    EPB = pd.concat([pd.read_csv(f"outputs_ews_{r}/metrik_episode.csv").assign(Resolusi=RES_NAME[r]) for r in D], ignore_index=True)
    EPB = EPB[EPB.Model.str.contains(" · ", regex=False)]                     # hanya model terpilih (nama pembanding tidak memuat " · ")
    show(EPB[["L (jam)", "Fase", "Resolusi", "Model", "Episode", "Deteksi (%)", "Lead time rata-rata (jam)", "Kejadian alarm palsu per minggu"]],
         {"L (jam)": "{:d}", "Episode": "{:d}", "Deteksi (%)": "{:.1f}", "Lead time rata-rata (jam)": "{:.2f}", "Kejadian alarm palsu per minggu": "{:.1f}"},
         "Tabel 14. Evaluasi per kejadian model terpilih tiap resolusi (deskriptif; origin tiap resolusi bisa sedikit berbeda)")
    # Gambar 5: PR-AUC dan MCC per resolusi, panel terpisah per fase
    fig, axs = plt.subplots(2, 2, figsize=(9.5, 5.6), sharex=True); x = np.arange(len(LEADS_B)); w = 0.25
    for i, ph in enumerate(("Validation", "Test")):
        for j, nm in enumerate(("PR-AUC", "MCC")):
            ax = axs[i, j]
            for k_, r in enumerate(sorted(D)):
                g = MTB[(MTB.Resolusi == RES_NAME[r]) & (MTB.Fase == ph)].set_index("L (jam)").reindex(LEADS_B)[nm]
                ax.bar(x + (k_ - 1) * w, g.to_numpy(), width=w * .9, color=COL[f"r{r}"], label=RES_NAME[r])
            ax.set_ylabel(nm); ax.set_title(f"{ph} · {nm} pada origin yang sama"); ax.set_ylim(0, 1 if nm == "PR-AUC" else 0.8)
            if i == 1: ax.set_xticks(x); ax.set_xticklabels([f"L = {L} jam" for L in LEADS_B])
    hd, lb = axs[0, 0].get_legend_handles_labels(); fig.legend(hd, lb, fontsize=8, ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.01))
    fig.suptitle("Gambar 5. Model EWS terpilih tiap resolusi, dinilai pada origin dan label yang sama", x=0.01, ha="left", fontsize=10)
    fig.tight_layout(rect=(0, 0.04, 1, 1)); fig.savefig(f"{OUT_BANDING}/figures/gambar5_tiga_resolusi.png", bbox_inches="tight"); plt.show()
    open(f"{OUT_BANDING}/ringkasan_banding.md", "w").write("\n".join(["# Perbandingan EWS tiga resolusi (origin yang sama, label per jam resmi)", "",
        "## Metrik model terpilih per resolusi", md_table(MTB[["L (jam)", "Fase", "Resolusi", "Model terpilih", "n", "positif"] + KOLOM_METRIK]), "",
        "## Selisih terhadap 1 jam, block bootstrap blok 7 hari (500 ulangan). Positif = resolusi itu lebih baik.", md_table(BT), "",
        "## Resolusi terbaik per jendela (PR-AUC validation)", md_table(best[["L (jam)", "Resolusi", "Model terpilih", "PR-AUC", "F1", "MCC"]])]))
else:
    print("Perbandingan dilewati: jalankan notebook ini untuk RES = 60, 30, dan 15 terlebih dahulu.")

## 10. Cara membaca hasil

- **Seleksi** hanya dari validation (Tabel 5–8). Test (Tabel 9–10) menunjukkan generalisasi ke periode yang lebih tenang (positif hanya sekitar 2 %), sehingga PR-AUC turun walaupun ROC-AUC tetap tinggi; itulah alasan PR-AUC dipakai sebagai metrik utama (Chicco & Jurman, 2023).
- **Penanganan ketidakseimbangan** dibaca dari Tabel 6 dan Gambar 1: bandingkan bobot kelas, SMOTE, dan hibrida terhadap "tanpa penanganan" pada ambang yang sudah disetel. Lihat PR-AUC **dan** Brier: bila PR-AUC setara tetapi Brier lebih buruk, koreksi itu hanya menggeser probabilitas tanpa menambah daya pembeda (van den Goorbergh dkk., 2022).
- **Seleksi** memakai aturan satu galat baku (Tabel 5 kolom "Dalam 1 SE", Tabel 8). Bila model terpilih bukan yang PR-AUC-nya tertinggi, itu disengaja: selisihnya masih di dalam derau sampel.
- **Nilai operasional EWS** dibaca dari Tabel 9b dan 10: apakah keunggulan atas ambang statis nyata (CI tidak memuat nol), berapa persen episode terdeteksi, berapa jam sebelumnya, dan berapa kejadian alarm palsu per minggu, dibandingkan dengan ambang statis yang lazim dipakai di sistem pemantauan.
- **Resolusi** dibandingkan pada origin yang sama (Tabel 11–13); kesimpulan hanya bila CI bootstrap tidak memuat nol.

## Daftar pustaka

Semua rujukan utama adalah artikel jurnal terindeks terbitan 2020–2026 (kecuali Chawla dkk., 2002 sebagai sumber algoritma SMOTE, dan buku James dkk., 2021). Kolom "Dipakai untuk" menunjukkan keputusan desain di notebook ini yang bersandar pada rujukan itu.

| # | Rujukan | Dipakai untuk |
|---|---|---|
| 1 | Li, Y., Jiang, Z. M., Li, H., Hassan, A. E., He, C., Huang, R., Zeng, Z., Wang, M., & Chen, P. (2020). Predicting node failures in an ultra-large-scale cloud computing platform: An AIOps solution. *ACM Transactions on Software Engineering and Methodology, 29*(2), Article 13. https://doi.org/10.1145/3385187 | EWS di infrastruktur cloud sebagai klasifikasi dengan **jendela prediksi** (2–72 jam): prediksi hanya dihitung berhasil bila berada di dalam jendela sebelum kejadian; **evaluasi time-based** (latih pada bulan sebelumnya, uji pada bulan berikutnya); data sangat tidak seimbang; AUC dipakai karena tidak bergantung ambang. |
| 2 | Hyland, S. L., Faltys, M., Hüser, M., Lyu, X., Gumbsch, T., Esteban, C., Bock, C., Horn, M., Moor, M., Rieck, B., Zimmermann, M., Bodenham, D., Borgwardt, K., Rätsch, G., & Merz, T. M. (2020). Early prediction of circulatory failure in the intensive care unit using machine learning. *Nature Medicine, 26*(3), 364–373. https://doi.org/10.1038/s41591-020-0789-4 | EWS dari deret waktu beresolusi tinggi dengan gradient boosting (LightGBM); label "kejadian dalam jendela ke depan" (3,1 % positif); **evaluasi berbasis kejadian**: persentase kejadian yang terdeteksi, lead time, dan laju alarm palsu per satuan waktu; pembanding sistem **ambang statis** konvensional; PR-AUC di samping ROC-AUC. |
| 3 | Gao, J., Wang, H., & Shen, H. (2022). Task failure prediction in cloud data centers using deep learning. *IEEE Transactions on Services Computing, 15*(3), 1411–1422. https://doi.org/10.1109/TSC.2020.2993728 | Prediksi kegagalan di pusat data cloud dari metrik telemetri (CPU, memori, disk) sebagai masalah klasifikasi terawasi. |
| 4 | Tengku Asmawi, T. N., Ismail, A., & Shen, J. (2022). Cloud failure prediction based on traditional machine learning and deep learning. *Journal of Cloud Computing, 11*, Article 47. https://doi.org/10.1186/s13677-022-00327-0 | Pemakaian **SMOTE** untuk kelas minoritas pada prediksi kegagalan cloud; pembandingan regresi logistik, random forest, gradient boosting, XGBoost, dan LSTM dengan presisi, sensitivitas, spesifisitas, dan F-score. Catatan: artikel ini tidak menyatakan apakah pemisahan datanya menjaga urutan waktu; notebook ini menjaganya. |
| 5 | Rezvani, S., & Wang, X. (2023). A broad review on class imbalance learning techniques. *Applied Soft Computing, 143*, 110415. https://doi.org/10.1016/j.asoc.2023.110415 | Taksonomi penanganan ketidakseimbangan: **tingkat data** (over/under-sampling), **tingkat algoritma** (cost-sensitive), dan **hibrida**. Grid strategi di notebook ini mengikuti taksonomi ini. |
| 6 | Thabtah, F., Hammoud, S., Kamalov, F., & Gonsalves, A. (2020). Data imbalance in classification: Experimental evaluation. *Information Sciences, 513*, 429–441. https://doi.org/10.1016/j.ins.2019.11.004 | Bukti empiris bahwa rasio ketidakseimbangan mengubah presisi, recall, dan AUC secara berbeda per dataset, sehingga strategi harus **dipilih secara empiris di validation**, bukan ditetapkan di muka. |
| 7 | Tarawneh, A. S., Hassanat, A. B., Altarawneh, G. A., & Almuhaimeed, A. (2022). Stop oversampling for class imbalance learning: A review. *IEEE Access, 10*, 47643–47660. https://doi.org/10.1109/ACCESS.2022.3169512 | Peringatan bahwa oversampling dapat menghasilkan sampel sintetis yang tidak mewakili kelas minoritas; karena itu "tanpa penanganan" dan cost-sensitive tetap diuji sebagai pembanding, dan resampling **hanya pada data training**. |
| 8 | Esposito, C., Landrum, G. A., Schneider, N., Stiefl, N., & Riniker, S. (2021). GHOST: Adjusting the decision threshold to handle imbalanced data in machine learning. *Journal of Chemical Information and Modeling, 61*(6), 2623–2640. https://doi.org/10.1021/acs.jcim.1c00160 | **Penyetelan ambang keputusan** (threshold moving) sebagai penanganan ketidakseimbangan tanpa resampling; ambang alarm di notebook ini dipilih di validation, bukan 0,5. |
| 9 | Moniz, N., & Cerqueira, V. (2021). Automated imbalanced classification via meta-learning. *Expert Systems with Applications, 178*, 115011. https://doi.org/10.1016/j.eswa.2021.115011 | Tidak ada satu strategi resampling yang terbaik untuk semua data; pemilihan strategi diperlakukan sebagai bagian dari seleksi model. |
| 10 | Chicco, D., & Jurman, G. (2020). The advantages of the Matthews correlation coefficient (MCC) over F1 score and accuracy in binary classification evaluation. *BMC Genomics, 21*, 6. https://doi.org/10.1186/s12864-019-6413-7 | **MCC** sebagai metrik ringkasan pada ambang terpilih. |
| 11 | Chicco, D., & Jurman, G. (2023). The Matthews correlation coefficient (MCC) should replace the ROC AUC as the standard metric for assessing binary classification. *BioData Mining, 16*, 4. https://doi.org/10.1186/s13040-023-00322-4 | ROC-AUC mengabaikan presisi dan terlalu optimistis pada kelas langka; seleksi memakai **PR-AUC**, ROC-AUC hanya dilaporkan. |
| 12 | Eldeeb, O. M., Aboutabl, A. E., & Bahloul, M. M. (2026). Imbalance-robust predictive maintenance using deep learning with virtual measurement modeling and explainable AI. *Scientific Reports, 16*, 30529. https://doi.org/10.1038/s41598-026-70467-9 | Perbandingan SMOTE saja, cost-sensitive saja, dan **hibrida SMOTE + cost-sensitive** pada LightGBM, XGBoost, dan lainnya; hibrida dilaporkan terbaik dan SMOTE saja menaikkan recall tetapi menurunkan presisi. Strategi "hibrida" di notebook ini mengikuti ini. |
| 13 | Cerqueira, V., Torgo, L., & Mozetič, I. (2020). Evaluating time series forecasting models: An empirical study on performance estimation methods. *Machine Learning, 109*, 1997–2028. https://doi.org/10.1007/s10994-020-05910-7 | Pemisahan data menjaga urutan waktu (tanpa shuffle, tanpa K-fold acak). |
| 14 | Hewamalage, H., Ackermann, K., & Bergmeir, C. (2023). Forecast evaluation for data scientists: Common pitfalls and best practices. *Data Mining and Knowledge Discovery, 37*(2), 788–832. https://doi.org/10.1007/s10618-022-00894-5 | Kebocoran data, label yang sudah matang sebelum refit, dan pembanding naif. |
| 16 | van den Goorbergh, R., van Smeden, M., Timmerman, D., & Van Calster, B. (2022). The harm of class imbalance corrections for risk prediction models: Illustration and simulation using logistic regression. *Journal of the American Medical Informatics Association, 29*(9), 1525–1534. https://doi.org/10.1093/jamia/ocac093 | Koreksi ketidakseimbangan (under/over-sampling, SMOTE) tidak menaikkan daya pembeda tetapi merusak **kalibrasi**; karena itu "tanpa penanganan" + ambang yang disetel tetap pembanding utama, Brier dilaporkan, dan aturan seleksi lebih memilih strategi yang lebih sederhana. |
| 17 | Mienye, I. D., & Sun, Y. (2022). A survey of ensemble learning: Concepts, algorithms, applications, and prospects. *IEEE Access, 10*, 99129–99149. https://doi.org/10.1109/ACCESS.2022.3207287 | Ensemble dengan merata-ratakan probabilitas beberapa model mengurangi varians dibanding model tunggal; dasar **ensemble GBDT**. |
| 18 | James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An introduction to statistical learning* (2nd ed.). Springer. https://doi.org/10.1007/978-1-0716-1418-1 (buku) | §6.1.3: **aturan satu galat baku** (one-standard-error rule): dari model yang galat validasinya dalam satu SE dari yang terbaik, pilih yang paling sederhana. |
| 15 | Chawla, N. V., Bowyer, K. W., Hall, L. O., & Kegelmeyer, W. P. (2002). SMOTE: Synthetic minority over-sampling technique. *Journal of Artificial Intelligence Research, 16*, 321–357. https://doi.org/10.1613/jair.953 | Algoritma SMOTE asli yang diimplementasikan di notebook ini (k = 5 tetangga, interpolasi acak). Rujukan sebelum 2020, dicantumkan sebagai sumber metode. |

## Pemetaan keputusan desain ke rujukan

| Keputusan di notebook ini | Rujukan |
|---|---|
| Label = CPU per jam akan ≥ 80 % dalam jendela L jam (default 2 jam); origin hanya saat CPU masih < 80 % | 1, 2 |
| Walk-forward refit bulanan, seleksi di validation, test sekali, tanpa shuffle | 1, 13, 14 |
| Empat strategi: tanpa, bobot kelas, SMOTE, hibrida; resampling hanya di training; under-sampling dihapus | 5, 6, 7, 12, 15, 16 |
| Ambang alarm disetel di validation (maks. F1), bukan 0,5 | 8 |
| Seleksi dengan PR-AUC dan aturan satu galat baku; MCC, presisi, recall, F1, spesifisitas, Brier dilaporkan | 10, 11, 2, 16, 18 |
| Evaluasi per kejadian: episode terdeteksi, lead time, alarm palsu per minggu; pembanding ambang statis | 2, 1 |
| Model: regresi logistik, HistGradientBoosting, LightGBM, XGBoost, ensemble GBDT | 2, 4, 12, 17 |


**Pustaka fitur dan protokol forecasting** (Januschowski dkk., 2022; Hyndman & Athanasopoulos, 2021; dan lainnya) ada di `skripsi_forecast_v3.ipynb`.